In [1]:
import os
import sys
sys.path.append(os.getcwd())

import polars as pl
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from mars.analysis.profiler import MarsDataProfiler
from mars.utils.logger import set_log_level

# 1. 设置环境
set_log_level("WARNING") 

def get_stress_test_data(rows=20000):
    """
    生成包含典型'脏数据'场景的测试集，并将时间轴升级为真实日期格式。
    """
    np.random.seed(42)
    
    # [升级] 生成真实日期序列 (覆盖 2024-01-01 到 2024-04-30)
    # 我们让前3个月相对稳定，第4个月发生漂移
    base_date = datetime(2024, 1, 1)
    date_range = [base_date + timedelta(days=x) for x in range(120)]
    
    # 随机采样日期
    dates = np.random.choice(date_range, size=rows)
    # 转换为 Polars Series 以便后续处理 (或保持 numpy array 也可以，Polars 都能吃)
    
    # 辅助列：为了生成漂移逻辑，我们需要知道每行数据属于哪个月
    # 这里简单处理：如果日期 >= 2024-04-01 则为漂移月
    is_drift_month = np.array([d >= datetime(2024, 4, 1) for d in dates])
    
    # --- A. 正常对照组 ---
    f_good = np.random.normal(100, 10, size=rows)

    # --- B. 数据质量 (DQ) 报警测试 ---
    
    # [Case 1] 缺失率爆炸 (Missing > 50%)
    f_missing = np.random.randn(rows)
    f_missing[np.random.rand(rows) < 0.6] = -999 
    
    # [Case 2] 唯一值爆炸 (High Cardinality)
    f_id = [f"uid_{i}" for i in range(rows)]
    
    # [Case 3] 单一值报警 (Zero Variance)
    f_const = np.zeros(rows)
    f_const[np.random.rand(rows) < 0.005] = 1 

    # --- C. 稳定性 (PSI) 报警测试 ---

    # [Case 4] 数值分布漂移 (PSI Explosion)
    # 前3个月均值0，第4个月均值突变为 5
    f_psi_num = np.random.normal(0, 1, size=rows)
    f_psi_num[is_drift_month] = np.random.normal(5, 1, size=np.sum(is_drift_month))

    # [Case 5] 特殊值占比漂移 (Special Value Shift)
    # 正常月份 1% 是 -1，漂移月份 50% 是 -1
    f_psi_special = np.random.normal(100, 10, size=rows)
    
    special_mask_normal = (~is_drift_month) & (np.random.rand(rows) < 0.01)
    f_psi_special[special_mask_normal] = -1
    
    special_mask_drift = (is_drift_month) & (np.random.rand(rows) < 0.50)
    f_psi_special[special_mask_drift] = -1

    return pl.DataFrame({
        "trans_date":dates.astype("datetime64[ms]"), 
        
        "feature_0_good": f_good,
        "feature_1_high_missing": f_missing,
        "feature_2_high_cardinality": f_id,
        "feature_3_quasi_constant": f_const,
        "feature_4_psi_drift": f_psi_num,
        "feature_5_special_drift": f_psi_special
    })

def run_verification():
    print("🚀 生成测试数据 (含真实日期列)...")
    df = get_stress_test_data()
    
    print("\n🧐 初始化 Profiler (配置特殊值逻辑)...")
    profiler = MarsDataProfiler(
        df, 
        custom_missing_values=[-999, "null"], 
        custom_special_values=[-1],
        exclude_features=["trans_date"]
    )

    print("📊 计算画像报告 (启用 Auto Date Aggregation)...")
    
    # ✅ [升级点] 使用新的调用方式
    # 我们的数据里没有 "month" 列，只有 "trans_date"
    # 我们要求 Profiler 自动按月聚合
    report = profiler.generate_profile(
        profile_by="week", 
        dt_col="trans_date"
    )
    
    # 获取原始数据进行断言验证
    overview, _, stats_tables = report.get_profile_data()
    psi_table = stats_tables.get('psi')

    print("\n✅ --- 验证结果 (Expectation vs Reality) ---")
    
    # 1.  6] Auto验证缺失率报警
    missing_rate = overview.filter(pl.col("feature") == "feature_1_high_missing")["missing_rate"][0]
    print(f"[Case 1] High Missing: Expected > 0.5 | Actual: {missing_rate:.2%} -> {'🔴 PASS' if missing_rate > 0.5 else '❌ FAIL'}")

    # 2. 验证常量报警
    top1 = overview.filter(pl.col("feature") == "feature_3_quasi_constant")["top1_ratio"][0]
    print(f"[Case 3] Constant:     Expected > 0.99| Actual: {top1:.2%} -> {'🔴 PASS' if top1 > 0.99 else '❌ FAIL'}")

    if psi_table is not None:
        # 3. 验证数值 PSI 报警
        psi_num = psi_table.filter(pl.col("feature") == "feature_4_psi_drift")["total"][0]
        print(f"[Case 4] Numeric PSI:  Expected > 0.25| Actual: {psi_num:.4f}  -> {'🔴 PASS' if psi_num > 0.25 else '❌ FAIL'}")

        # 4. 验证特殊值 PSI 报警
        psi_special = psi_table.filter(pl.col("feature") == "feature_5_special_drift")["total"][0]
        print(f"[Case 5] Special PSI:  Expected > 0.1 | Actual: {psi_special:.4f}  -> {'🔴 PASS' if psi_special > 0.1 else '❌ FAIL'}")
        
        # [新增] 验证自动聚合是否成功
        # 检查 PSI 表的列名里是否包含自动生成的日期列 (例如 "2024-04-01")
        cols = psi_table.columns
        has_date_col = any("2024-" in c for c in cols)
        print(f"[Case Aggregation: Check if date columns exist -> {'🟢 PASS' if has_date_col else '❌ FAIL'}")
    
    return report


report = run_verification()
# report.write_excel("stress_test_report.xlsx") # 可选：导出查看效果

🚀 生成测试数据 (含真实日期列)...

🧐 初始化 Profiler (配置特殊值逻辑)...
📊 计算画像报告 (启用 Auto Date Aggregation)...

✅ --- 验证结果 (Expectation vs Reality) ---
[Case 1] High Missing: Expected > 0.5 | Actual: 59.82% -> 🔴 PASS
[Case 3] Constant:     Expected > 0.99| Actual: 99.44% -> 🔴 PASS
[Case 4] Numeric PSI:  Expected > 0.25| Actual: 2.3705  -> 🔴 PASS
[Case 5] Special PSI:  Expected > 0.1 | Actual: 0.2811  -> 🔴 PASS
[Case Aggregation: Check if date columns exist -> 🟢 PASS


In [2]:
report

In [3]:
report.show_overview()

feature,dtype,distribution,missing_rate,zeros_rate,unique_rate,top1_ratio,top1_value,mean,std,min,max,p25,median,p75,skew,kurtosis
feature_0_good,Float64,▂▂▄█▆▃▂▂,0.00%,0.00%,100.00%,0.01%,94.21787901780246,100.03,9.93,60.78,144.79,93.24,100.07,106.74,-0.00,-0.01
feature_1_high_missing,Float64,▂▂▃▆█▅▂▂,59.82%,0.00%,40.18%,59.82%,-999.0,0.00,1.00,-4.47,3.60,-0.68,-0.00,0.68,-0.02,-0.00
feature_3_quasi_constant,Float64,█______▂,0.00%,99.44%,0.01%,99.44%,0.0,0.01,0.07,0.00,1.00,0.00,0.00,0.00,13.25,173.58
feature_4_psi_drift,Float64,▂▄█▃▂▃▃▂,0.00%,0.00%,100.00%,0.01%,3.690991382837683,1.19,2.36,-3.71,8.55,-0.45,0.40,2.16,0.91,-0.35
feature_5_special_drift,Float64,▂▂▄█▇▃▂▂,0.00%,0.00%,87.42%,12.58%,-1.0,100.04,9.97,60.01,142.02,93.34,100.08,106.69,-0.01,0.07
feature_2_high_cardinality,String,None,0.00%,0.00%,100.00%,0.01%,uid_0,nan,nan,nan,nan,nan,nan,nan,nan,nan


In [4]:
report.show_dq("missing")

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total
feature_0_good,Float64,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
feature_1_high_missing,Float64,59.57%,59.72%,60.96%,59.08%,59.11%,60.52%,58.20%,55.99%,58.65%,61.10%,59.09%,60.30%,59.46%,61.27%,58.79%,62.35%,62.29%,64.02%,59.82%
feature_3_quasi_constant,Float64,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
feature_4_psi_drift,Float64,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
feature_5_special_drift,Float64,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
feature_2_high_cardinality,String,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%


In [5]:
report.show_dq("zeros")

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total
feature_0_good,Float64,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
feature_1_high_missing,Float64,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
feature_3_quasi_constant,Float64,99.37%,99.57%,99.60%,99.58%,99.46%,99.57%,99.32%,99.14%,99.59%,99.47%,99.48%,99.32%,99.58%,99.05%,99.56%,99.14%,99.66%,99.39%,99.44%
feature_4_psi_drift,Float64,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
feature_5_special_drift,Float64,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
feature_2_high_cardinality,String,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%


In [6]:
report.show_dq("unique")

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total
feature_0_good,Float64,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%
feature_1_high_missing,Float64,40.52%,40.36%,39.12%,41.00%,40.98%,39.57%,41.89%,44.09%,41.43%,38.99%,40.99%,39.78%,40.62%,38.82%,41.29%,37.74%,37.80%,36.59%,40.18%
feature_3_quasi_constant,Float64,0.18%,0.17%,0.16%,0.17%,0.18%,0.17%,0.17%,0.17%,0.16%,0.18%,0.17%,0.17%,0.17%,0.17%,0.17%,0.17%,0.17%,1.22%,0.01%
feature_4_psi_drift,Float64,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%
feature_5_special_drift,Float64,98.83%,98.88%,98.88%,99.41%,99.20%,99.22%,98.39%,98.89%,98.69%,98.85%,99.39%,98.90%,99.41%,51.56%,49.69%,51.55%,51.12%,50.61%,87.42%
feature_2_high_cardinality,String,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%


In [7]:
report.show_dq("top1")

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total
feature_0_good,Float64,0.09%,0.09%,0.08%,0.08%,0.09%,0.09%,0.08%,0.09%,0.08%,0.09%,0.09%,0.08%,0.08%,0.09%,0.09%,0.09%,0.09%,0.61%,0.01%
feature_1_high_missing,Float64,59.57%,59.72%,60.96%,59.08%,59.11%,60.52%,58.20%,55.99%,58.65%,61.10%,59.09%,60.30%,59.46%,61.27%,58.79%,62.35%,62.29%,64.02%,59.82%
feature_3_quasi_constant,Float64,99.37%,99.57%,99.60%,99.58%,99.46%,99.57%,99.32%,99.14%,99.59%,99.47%,99.48%,99.32%,99.58%,99.05%,99.56%,99.14%,99.66%,99.39%,99.44%
feature_4_psi_drift,Float64,0.09%,0.09%,0.08%,0.08%,0.09%,0.09%,0.08%,0.09%,0.08%,0.09%,0.09%,0.08%,0.08%,0.09%,0.09%,0.09%,0.09%,0.61%,0.01%
feature_5_special_drift,Float64,1.26%,1.20%,1.20%,0.67%,0.89%,0.86%,1.70%,1.20%,1.39%,1.24%,0.70%,1.18%,0.67%,48.53%,50.39%,48.53%,48.97%,50.00%,12.58%
feature_2_high_cardinality,String,0.09%,0.09%,0.08%,0.08%,0.09%,0.09%,0.08%,0.09%,0.08%,0.09%,0.09%,0.08%,0.08%,0.09%,0.09%,0.09%,0.09%,0.61%,0.01%


In [8]:
report.show_trend("psi")

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total,group_mean,group_var,group_cv
feature_0_good,Float64,0.00,0.01,0.01,0.01,0.01,0.00,0.01,0.01,0.02,0.00,0.01,0.01,0.01,0.02,0.00,0.01,0.02,0.03,0.00,0.01,0.0001,0.0000
feature_1_high_missing,Float64,0.00,0.01,0.01,0.01,0.01,0.02,0.01,0.01,0.02,0.02,0.01,0.02,0.01,0.02,0.02,0.03,0.02,0.10,0.01,0.02,0.0005,1.0906
feature_2_high_cardinality,String,0.00,13.55,13.47,13.52,13.58,13.55,13.53,13.54,13.50,13.57,13.56,13.53,13.52,13.55,13.56,13.55,13.55,15.51,6.35,12.90,10.5768,0.2522
feature_3_quasi_constant,Float64,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0000,0.0000
feature_4_psi_drift,Float64,0.00,0.01,0.01,0.01,0.01,0.01,0.02,0.01,0.02,0.00,0.01,0.01,0.01,20.95,21.70,21.58,21.57,22.34,2.37,6.01,99.3024,1.6570
feature_5_special_drift,Float64,0.00,0.01,0.00,0.03,0.01,0.03,0.02,0.02,0.01,0.01,0.01,0.02,0.03,2.05,2.17,2.05,2.08,2.17,0.28,0.60,0.9256,1.6145


In [9]:
report.show_trend("mean")

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total,group_mean,group_var,group_cv
feature_0_good,Float64,99.95,100.10,100.31,100.24,99.63,100.05,99.92,100.32,100.71,99.71,99.93,100.05,100.17,99.76,99.76,99.62,100.26,99.44,100.03,100.00,0.0988,0.0031
feature_1_high_missing,Float64,0.01,-0.04,-0.06,0.06,-0.01,-0.04,-0.00,0.01,0.05,0.07,-0.00,-0.07,-0.03,0.02,0.04,-0.05,0.02,0.06,0.00,0.00,0.0019,14.7381
feature_3_quasi_constant,Float64,0.01,0.00,0.00,0.00,0.01,0.00,0.01,0.01,0.00,0.01,0.01,0.01,0.00,0.01,0.00,0.01,0.00,0.01,0.01,0.01,0.0000,0.3207
feature_4_psi_drift,Float64,0.02,-0.00,0.03,-0.02,0.03,-0.03,0.04,-0.02,-0.01,-0.01,0.01,0.00,-0.07,5.03,5.01,4.98,4.99,5.06,1.19,1.39,5.3450,1.6619
feature_5_special_drift,Float64,99.97,100.11,100.09,100.23,99.80,100.19,100.20,99.27,100.29,99.87,100.25,100.36,99.93,99.93,99.94,99.74,100.41,99.53,100.04,100.01,0.0874,0.0030
feature_2_high_cardinality,String,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,0.00,0.0000,0.0000


In [10]:
report.show_trend("max")

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total,group_mean,group_var,group_cv
feature_0_good,Float64,132.15,139.42,136.11,129.45,129.14,133.78,128.74,128.22,144.79,129.57,132.88,129.70,132.86,131.37,129.38,136.03,134.29,129.97,144.79,132.66,18.5979,0.0325
feature_1_high_missing,Float64,2.86,2.94,2.99,3.01,3.00,3.28,2.35,2.90,3.16,3.10,2.96,3.60,3.19,2.48,2.78,2.23,2.64,2.68,3.60,2.90,0.1135,0.1163
feature_3_quasi_constant,Float64,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.0000,0.0000
feature_4_psi_drift,Float64,3.71,3.56,3.69,3.31,2.98,3.59,3.26,3.52,3.06,3.03,3.83,3.71,2.90,8.11,8.21,8.55,8.13,7.44,8.55,4.70,4.7831,0.4654
feature_5_special_drift,Float64,133.89,137.66,129.29,129.49,133.76,132.61,135.75,142.02,133.84,136.00,131.83,134.28,131.81,130.83,132.28,128.28,131.13,127.23,142.02,132.89,12.7293,0.0268
feature_2_high_cardinality,String,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,0.00,0.0000,0.0000


In [11]:
report.show_trend("min")

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total,group_mean,group_var,group_cv
feature_0_good,Float64,61.44,66.79,66.64,63.65,60.78,70.27,67.04,70.46,70.32,66.70,67.87,66.65,70.66,65.05,66.24,69.81,63.45,73.22,60.78,67.06,11.3326,0.0502
feature_1_high_missing,Float64,-2.89,-2.71,-2.92,-2.43,-4.47,-2.87,-2.93,-3.16,-3.63,-2.51,-3.20,-2.63,-3.00,-3.53,-3.43,-2.98,-3.22,-2.69,-4.47,-3.07,0.2333,0.1575
feature_3_quasi_constant,Float64,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0000,0.0000
feature_4_psi_drift,Float64,-2.91,-2.83,-3.15,-3.04,-3.14,-2.90,-2.99,-2.80,-2.82,-3.43,-3.71,-3.01,-3.13,1.37,1.69,1.75,2.10,2.50,-3.71,-1.69,5.2911,1.3597
feature_5_special_drift,Float64,65.35,68.85,62.99,70.96,70.87,66.79,62.17,68.03,68.75,69.13,64.68,69.17,68.00,71.43,60.01,72.90,68.92,65.84,60.01,67.49,11.7664,0.0508
feature_2_high_cardinality,String,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,0.00,0.0000,0.0000


In [12]:
report.show_trend('skew')

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total,group_mean,group_var,group_cv
feature_0_good,Float64,-0.11,0.07,0.01,-0.07,-0.09,0.00,0.03,0.01,0.05,-0.06,-0.06,-0.03,0.05,-0.09,-0.06,0.14,0.09,0.09,-0.00,-0.00,0.0052,27.5130
feature_1_high_missing,Float64,0.07,0.08,0.11,0.13,-0.18,-0.03,-0.14,0.07,0.02,0.05,-0.06,0.26,-0.04,-0.11,-0.11,-0.20,-0.22,-0.25,-0.02,-0.03,0.0196,4.5471
feature_3_quasi_constant,Float64,12.46,15.15,15.72,15.36,13.55,15.13,12.01,10.67,15.52,13.62,13.73,12.04,15.36,10.10,15.02,10.62,16.97,12.69,13.25,13.65,4.0757,0.1479
feature_4_psi_drift,Float64,0.04,0.21,0.01,0.02,-0.05,0.04,-0.03,0.08,0.07,-0.09,-0.03,0.08,-0.07,-0.04,-0.04,-0.08,-0.05,-0.07,0.91,0.00,0.0056,2764.5446
feature_5_special_drift,Float64,-0.09,-0.02,-0.17,-0.01,0.15,-0.06,-0.03,-0.04,-0.06,0.10,-0.01,0.09,-0.02,0.13,-0.11,0.10,0.00,-0.20,-0.01,-0.01,0.0096,7.2990
feature_2_high_cardinality,String,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,0.00,0.0000,0.0000


In [13]:
report.write_excel("mars_demo_report.xlsx")

In [15]:
overview, dq_tables, stat_tables = report.get_profile_data()
overview

feature,dtype,distribution,missing_rate,zeros_rate,unique_rate,top1_ratio,top1_value,mean,std,min,max,p25,median,p75,skew,kurtosis
str,str,str,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""feature_0_good""","""Float64""","""▂▂▄█▆▃▂▂""",0.0,0.0,1.0,0.00005,"""94.21787901780246""",100.030674,9.930353,60.775997,144.790843,93.244315,100.065479,106.743678,-0.004967,-0.012534
"""feature_1_high_missing""","""Float64""","""▂▂▃▆█▅▂▂""",0.59825,0.0,0.4018,0.59825,"""-999.0""",0.000107,0.997207,-4.465604,3.602415,-0.677429,-0.001404,0.676057,-0.018504,-0.002242
"""feature_3_quasi_constant""","""Float64""","""█______▂""",0.0,0.9944,0.0001,0.9944,"""0.0""",0.0056,0.074625,0.0,1.0,0.0,0.0,0.0,13.250549,173.57706
"""feature_4_psi_drift""","""Float64""","""▂▄█▃▂▃▃▂""",0.0,0.0,1.0,0.00005,"""3.690991382837683""",1.194784,2.357119,-3.70515,8.548682,-0.447,0.398088,2.159562,0.910374,-0.347223
"""feature_5_special_drift""","""Float64""","""▂▂▄█▇▃▂▂""",0.0,0.0,0.87425,0.1258,"""-1.0""",100.037999,9.969301,60.006678,142.020259,93.335162,100.078962,106.692123,-0.010999,0.068424
"""feature_2_high_cardinality""","""String""",null,0.0,0.0,1.0,0.00005,"""uid_0""",null,null,null,null,null,null,null,null,null


In [16]:
stat_tables["psi"]

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total,group_mean,group_var,group_cv
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""feature_0_good""","""Float64""",0.0,0.005403,0.010755,0.007192,0.010564,0.003489,0.012181,0.012814,0.015567,0.003144,0.012055,0.008703,0.008663,0.017019,0.001748,0.014894,0.015221,0.030465,0.002684,0.010549,0.00005,0.0
"""feature_1_high_missing""","""Float64""",0.0,0.014203,0.013543,0.012649,0.014034,0.02399,0.00624,0.011481,0.019354,0.016598,0.011624,0.017947,0.01047,0.017669,0.019026,0.025361,0.017653,0.102259,0.007468,0.019672,0.00046,1.090607
"""feature_2_high_cardinality""","""String""",0.0,13.547419,13.473914,13.519226,13.584475,13.549153,13.534506,13.542234,13.499204,13.574638,13.558744,13.528536,13.520069,13.554373,13.564015,13.55089,13.545688,15.511873,6.354285,12.89772,10.576829,0.252153
"""feature_3_quasi_constant""","""Float64""",0.0,1.7612e-27,1.0512e-26,4.3198e-27,9.4367e-29,1.6331e-27,2.7928e-27,2.1485e-27,6.7497e-27,3.3761e-28,1.0366e-27,3.3586e-27,4.2255e-27,1.2939e-27,7.6729e-28,1.5229e-27,1.8892e-27,2.6988e-23,7.2664e-25,1.5018e-24,4.0455e-47,0.0
"""feature_4_psi_drift""","""Float64""",0.0,0.007389,0.011509,0.010133,0.009661,0.011287,0.015936,0.007241,0.015874,0.00303,0.010078,0.014986,0.010743,20.945407,21.702397,21.576078,21.565714,22.335282,2.370509,6.014041,99.302421,1.656966
"""feature_5_special_drift""","""Float64""",0.0,0.012847,0.00262,0.029048,0.013351,0.02875,0.018691,0.023231,0.012494,0.01428,0.014246,0.017492,0.025092,2.053402,2.165251,2.046018,2.082164,2.167096,0.281062,0.595893,0.925617,1.614535


In [17]:
# 伪代码逻辑
def filter_bad_features(df):
    
    # 1. 删掉“太无聊”的列 (准常量)
    # 只要 Top1 占比太高，不管它是啥类型，直接删
    top1_ratio = 0.96
    if top1_ratio > 0.95: 
        return "DROP (Quasi-Constant)"

    # 2. 删掉“太杂乱”的 ID 列 (高基数)
    # 只有当它是字符串类型，且几乎都不重复时，才删
    # (数值型的高Unique是好事，不能删！)
    unique_rate = 1.0
    is_string_type = True
    if unique_rate > 0.95 and is_string_type:
        return "DROP (ID Column)"
        
    return "KEEP"

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import time
import psutil
import os
import gc
import sys
from typing import Tuple, Union

from mars.analysis.profiler import MarsDataProfiler
from mars.utils.logger import set_log_level, logger

# ==========================================
# ⚙️ Stress Test Configuration
# ==========================================
# set_log_level("WARNING")  # Reduce IO interference

# Standard medium-sized risk dataset configuration (200k rows x 2000 cols)
N_ROWS: int = 200000
N_COLS: int = 5000
N_CATS: int = 50
N_GROUPS: int = 12

class Colors:
    GREEN = '\033[92m'
    RED = '\033[91m'
    CYAN = '\033[96m'
    BOLD = '\033[1m'
    RESET = '\033[0m'

def get_memory_usage() -> float:
    """Get current process memory (MB)"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

def generate_data() -> pl.DataFrame:
    """Fast large-scale test data generation"""
    print(f"{Colors.CYAN}🚀 [DataGen] Generating {N_ROWS:,} rows x {N_COLS} cols...{Colors.RESET}")
    start = time.time()
    
    # 1. Numerical columns (Matrix generation is faster)
    n_num = N_COLS - N_CATS
    # Using float32 to save memory for the demo, can be float64
    data = (np.random.randn(N_ROWS, n_num).astype(np.float32) * 10) + 100
    
    # Inject missing values (-999)
    mask = np.random.random(data.shape) < 0.1
    data[mask] = -999
    
    data_dict = {f"num_{i}": data[:, i] for i in range(n_num)}
    
    # 2. Categorical columns
    cats = ["A", "B", "C", "D", "E", "unknown", None]
    for i in range(N_CATS):
        data_dict[f"cat_{i}"] = np.random.choice(cats, size=N_ROWS).tolist()
        
    # 3. Group column
    groups = [f"2023{m:02d}" for m in range(1, N_GROUPS + 1)]
    data_dict["month"] = np.random.choice(groups, size=N_ROWS).tolist()
    
    df = pl.DataFrame(data_dict)
    size_mb = df.estimated_size('mb')
    print(f"✅ Data Ready! Size: {size_mb:.2f} MB | Time: {time.time()-start:.2f}s")
    
    # Check if memory is sufficient for conversion later
    if size_mb * 3 > psutil.virtual_memory().available / 1024 / 1024:
        print(f"{Colors.RED}⚠️ Warning: Dataset might be too large for Pandas conversion on this machine.{Colors.RESET}")
        
    return df

def run_benchmark_round(df: Union[pl.DataFrame, pd.DataFrame], backend: str) -> Tuple[float, float, float]:
    """Execute a single round of stress testing"""
    print(f"\n🔹 Testing Backend: {Colors.BOLD}{backend}{Colors.RESET}")
    print("-" * 60)
    
    # Configuration overrides for benchmark (disable sparklines for pure calc speed)
    # If you want to test sparkline performance, remove "enable_sparkline": False
    bench_config = {
        "enable_sparkline": True,
        # "stat_metrics": ["mean", "std", "min", "max", "p25", "median", "p75", "skew", "kurtosis"]
        } 

    # 1. Initialization
    gc.collect()
    t0 = time.time()
    # Initialize Engine
    profiler = MarsDataProfiler(df, custom_missing_values=[-999, "unknown"])
    t_init = time.time() - t0
    print(f"   1. Init Engine       : {t_init:.4f} s")
    
    # 2. Overview Only (Simulates old get_report)
    # Using profile_by=None to get global stats
    t1 = time.time()
    _ = profiler.generate_profile(profile_by=None, config_overrides=bench_config)
    t_report = time.time() - t1
    print(f"   2. Full Overview     : {t_report:.4f} s")
    
    # 3. Group Profile (Stability Analysis)
    # Using profile_by="month"
    t2 = time.time()
    _ = profiler.generate_profile(
        profile_by="month", 
        config_overrides=bench_config # Reuse config to keep sparklines off
    )
    t_profile = time.time() - t2
    print(f"   3. Group Profile (by): {t_profile:.4f} s")
    
    return t_init, t_report, t_profile

def print_final_report(pl_times, pd_times):
    """Print comparison report"""
    stages = ["Initialization", "Get Full Overview", "Generate Group Profile"]
    
    print(f"\n{Colors.BOLD}{'🏆 BENCHMARK RESULTS (Time in Seconds)':^65}{Colors.RESET}")
    print("=" * 65)
    print(f"| {'Stage':<24} | {'Polars':<10} | {'Pandas':<10} | {'Speedup':<10} |")
    print("|" + "-"*26 + "+" + "-"*12 + "+" + "-"*12 + "+" + "-"*12 + "|")
    
    for stage, t_pl, t_pd in zip(stages, pl_times, pd_times):
        # Calculate speedup
        speedup = t_pd / t_pl if t_pl > 0 else 0
        
        # Color code the winner
        if t_pl < t_pd:
            pl_str = f"{Colors.GREEN}{t_pl:.4f}{Colors.RESET}"
            pd_str = f"{t_pd:.4f}"
        else:
            pl_str = f"{t_pl:.4f}"
            pd_str = f"{Colors.GREEN}{t_pd:.4f}{Colors.RESET}"
            
        print(f"| {stage:<24} | {pl_str:<19} | {pd_str:<19} | {speedup:>9.1f}x |")
    print("=" * 65)
    print(f"{Colors.CYAN}* Speedup > 1.0x means Polars is faster.{Colors.RESET}")
    print(f"{Colors.CYAN}* Note: Sparklines were disabled for pure calculation benchmark.{Colors.RESET}\n")

# 1. Generate Data (Polars Native)
df_pl = generate_data()

# 2. Run Polars Benchmark
pl_results = run_benchmark_round(df_pl, "Polars (Native)")

# 3. Convert to Pandas for Comparison
print(f"\n{Colors.CYAN}🔄 Converting to Pandas for compatibility test...{Colors.RESET}")
try:
    t_conv = time.time()
    df_pd = df_pl.to_pandas()
    print(f"   Conversion time: {time.time() - t_conv:.2f}s")
    
    # Explicitly delete Polars DF to free memory if tight
    # del df_pl 
    gc.collect()
except MemoryError:
    print(f"{Colors.RED}❌ OOM during conversion! Skipping Pandas test.{Colors.RESET}")
    sys.exit(1)
    
# 4. Run Pandas Benchmark
pd_results = run_benchmark_round(df_pd, "Pandas (Compat)")  

# 5. Show Summary
print_final_report(pl_results, pd_results)

🚀 [DataGen] Generating 200,000 rows x 5000 cols...
✅ Data Ready! Size: 3794.04 MB | Time: 27.71s

🔹 Testing Backend: Polars (Native)
------------------------------------------------------------
   1. Init Engine       : 0.0000 s
[MARS] 2026-01-20 22:19:37 - INFO - ⏱️ [MarsDataProfiler.generate_profile] finished in 33.2908s
   2. Full Overview     : 33.2908 s
